# Predicción del tiempo de entrega de pedidos de comida

## 01 - Comprensión inicial y evaluación de la calidad de los datos

### Objetivo del proyecto

El objetivo de este proyecto es analizar los factores operativos, temporales y contextuales que influyen en el tiempo de entrega de pedidos de comida.

A partir del análisis exploratorio de los datos, se buscará identificar patrones y relaciones relevantes entre las distintas variables del dataset y el tiempo de entrega.

Posteriormente, se evaluará la posibilidad de desarrollar modelos de Machine Learning capaces de predecir el tiempo de entrega a partir de las características disponibles.

### Objetivo de este notebook

En esta primera etapa se realizará una comprensión inicial del dataset, incluyendo:

- carga de los datos;
- análisis de su estructura y dimensiones;
- identificación de los tipos de variables;
- evaluación de valores faltantes;
- detección de registros duplicados;
- búsqueda de posibles inconsistencias;
- análisis inicial de la variable objetivo.

Esta etapa permitirá evaluar la calidad de los datos antes de comenzar el análisis exploratorio y el posterior modelado.

## 1. Importación de librerías

Se importan las librerías necesarias para la manipulación, análisis y posterior visualización de los datos.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Configuración de visualización de pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Carga del dataset

Se utiliza una ruta relativa para acceder al archivo original almacenado en `data/raw`.

El uso de rutas relativas permite que el proyecto sea portable y reproducible, evitando depender de una ubicación específica dentro de la computadora utilizada para desarrollar el análisis.

In [2]:
DATA_PATH = Path("../data/raw/Food_Delivery_Time_Prediction.csv")

print(f"Ruta del dataset: {DATA_PATH}")
print(f"¿El archivo existe?: {DATA_PATH.exists()}")

Ruta del dataset: ..\data\raw\Food_Delivery_Time_Prediction.csv
¿El archivo existe?: True


### Lectura del archivo

A continuación, se carga el archivo CSV utilizando pandas y se verifican las dimensiones iniciales del dataset.

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset cargado correctamente.")
print(f"Cantidad de filas: {df.shape[0]:,}")
print(f"Cantidad de columnas: {df.shape[1]}")

Dataset cargado correctamente.
Cantidad de filas: 50,000
Cantidad de columnas: 24


## 3. Inspección inicial de los datos

Antes de realizar cualquier transformación o limpieza, se examina la estructura original del dataset.

Esta inspección permite conocer las variables disponibles, observar ejemplos de registros y detectar inicialmente posibles problemas de calidad de datos.

In [4]:
df.head()

,Order_ID,Order_Date,Order_Hour,Day_of_Week,Is_Weekend,Is_Festival,Weather,Pickup_Zone,Dropoff_Zone,Vehicle_Type,Rider_Experience_Years,Rider_Rating,Restaurant_Rating,Cuisine_Type,Order_Items,Restaurant_Load,Preparation_Time_Min,Road_Distance_km,Delivery_Distance_Category,Traffic_Level,Number_of_Signals,Average_Speed_kmph,Delivery_Priority,Time_taken_min
0,ORD000001,2025-10-28,8,Tuesday,0,0,Clear,CBD,Commercial,Scooter,3.1,3.8,3.8,Pizza,1,Medium,23,30.55,Long,Moderate,15,28.7,Normal,91
1,ORD000002,2025-09-26,12,Friday,0,0,Rain,CBD,CBD,Scooter,1.8,3.6,3.9,Desserts,3,Medium,11,2.25,Short,Moderate,17,23.3,Normal,21
2,ORD000003,2025-01-26,18,Sunday,1,0,Clear,Residential,Residential,Scooter,3.8,3.9,4.0,Biryani,1,Low,31,10.15,Long,Moderate,8,28.9,Normal,52
3,ORD000004,2025-11-19,12,Wednesday,0,0,Cloudy,Residential,Industrial,Bike,1.7,3.9,4.6,Burger,3,Medium,19,26.33,Long,Low,9,41.4,Normal,59
4,ORD000005,2025-08-04,18,Monday,0,0,Rain,Industrial,Residential,Bike,12.4,4.4,4.0,Desserts,1,Medium,7,30.48,Long,Moderate,8,31.7,Normal,75


In [5]:
print(f"Dimensiones del dataset: {df.shape}")

Dimensiones del dataset: (50000, 24)


In [6]:
df.columns.tolist()

['Order_ID',
 'Order_Date',
 'Order_Hour',
 'Day_of_Week',
 'Is_Weekend',
 'Is_Festival',
 'Weather',
 'Pickup_Zone',
 'Dropoff_Zone',
 'Vehicle_Type',
 'Rider_Experience_Years',
 'Rider_Rating',
 'Restaurant_Rating',
 'Cuisine_Type',
 'Order_Items',
 'Restaurant_Load',
 'Preparation_Time_Min',
 'Road_Distance_km',
 'Delivery_Distance_Category',
 'Traffic_Level',
 'Number_of_Signals',
 'Average_Speed_kmph',
 'Delivery_Priority',
 'Time_taken_min']

In [7]:
df.sample(5, random_state=42)

,Order_ID,Order_Date,Order_Hour,Day_of_Week,Is_Weekend,Is_Festival,Weather,Pickup_Zone,Dropoff_Zone,Vehicle_Type,Rider_Experience_Years,Rider_Rating,Restaurant_Rating,Cuisine_Type,Order_Items,Restaurant_Load,Preparation_Time_Min,Road_Distance_km,Delivery_Distance_Category,Traffic_Level,Number_of_Signals,Average_Speed_kmph,Delivery_Priority,Time_taken_min
33553,ORD033554,2025-03-22,6,Saturday,1,0,Fog,Commercial,Suburban,Bicycle,5.2,3.8,3.8,Pizza,5,High,38,19.89,Long,Low,12,14.8,Normal,120
9427,ORD009428,2025-07-25,17,Friday,0,0,Rain,CBD,Suburban,Electric Scooter,3.0,3.9,3.8,South Indian,5,Medium,29,37.57,Long,Moderate,17,20.8,Normal,145
199,ORD000200,2025-10-31,21,Friday,0,1,Rain,Residential,CBD,Bike,5.9,4.1,4.7,Cafe,5,Low,16,19.77,Long,High,10,23.7,Normal,79
12447,ORD012448,2025-10-10,3,Friday,0,0,Clear,Suburban,Industrial,Bicycle,9.0,4.7,4.8,Pizza,1,Medium,22,5.77,Medium,Low,2,17.4,Normal,44
39489,ORD039490,2025-09-02,5,Tuesday,0,0,Clear,Commercial,CBD,Bike,3.6,3.8,4.1,North Indian,5,Medium,32,27.49,Long,Low,10,43.5,Normal,71


### Estructura del dataset

Se utiliza `info()` para analizar los nombres de las columnas, la cantidad de valores no nulos y los tipos de datos asignados inicialmente por pandas.

Esta información permitirá detectar variables cuyo tipo de dato pueda requerir alguna transformación posterior.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Order_ID                    50000 non-null  str    
 1   Order_Date                  50000 non-null  str    
 2   Order_Hour                  50000 non-null  int64  
 3   Day_of_Week                 50000 non-null  str    
 4   Is_Weekend                  50000 non-null  int64  
 5   Is_Festival                 50000 non-null  int64  
 6   Weather                     50000 non-null  str    
 7   Pickup_Zone                 50000 non-null  str    
 8   Dropoff_Zone                50000 non-null  str    
 9   Vehicle_Type                50000 non-null  str    
 10  Rider_Experience_Years      50000 non-null  float64
 11  Rider_Rating                50000 non-null  float64
 12  Restaurant_Rating           50000 non-null  float64
 13  Cuisine_Type                50000 non-null

## 4. Evaluación de calidad e integridad de los datos

Antes de realizar el análisis exploratorio, se evalúa la calidad del dataset original.

Se analizarán valores faltantes, registros duplicados, tipos de datos, cardinalidad de las variables y posibles inconsistencias. El objetivo es identificar los problemas existentes antes de tomar decisiones de limpieza o transformación.

### 4.1 Valores faltantes

Se cuantifica la cantidad y el porcentaje de valores faltantes por variable. Analizar tanto la frecuencia absoluta como relativa permite evaluar la magnitud del problema y definir posteriormente una estrategia adecuada para cada variable.

In [9]:
valores_faltantes = pd.DataFrame({
    "Cantidad_nulos": df.isnull().sum(),
    "Porcentaje_nulos": (df.isnull().mean() * 100).round(2)
})

valores_faltantes = valores_faltantes.sort_values(
    by="Porcentaje_nulos",
    ascending=False
)

valores_faltantes

,Cantidad_nulos,Porcentaje_nulos
Order_ID,0,0.0
Order_Date,0,0.0
Order_Hour,0,0.0
Day_of_Week,0,0.0
Is_Weekend,0,0.0
Is_Festival,0,0.0
Weather,0,0.0
Pickup_Zone,0,0.0
Dropoff_Zone,0,0.0
Vehicle_Type,0,0.0


In [10]:
print(f"Total de valores faltantes: {df.isnull().sum().sum():,}")
print(
    f"Filas con al menos un valor faltante: "
    f"{df.isnull().any(axis=1).sum():,}"
)

Total de valores faltantes: 0
Filas con al menos un valor faltante: 0


### 4.2 Registros duplicados

Se verifica la existencia de registros completamente duplicados. Los duplicados pueden afectar las distribuciones observadas y otorgar un peso artificial a determinadas observaciones durante el análisis y el posterior modelado.

In [11]:
duplicados = df.duplicated().sum()

print(f"Cantidad de registros duplicados: {duplicados:,}")
print(f"Porcentaje de registros duplicados: {duplicados / len(df) * 100:.2f}%")

Cantidad de registros duplicados: 0
Porcentaje de registros duplicados: 0.00%


### 4.3 Tipos de datos

Se revisan los tipos de datos asignados por pandas para comprobar si son coherentes con la naturaleza de cada variable.

Esta revisión resulta especialmente importante para detectar fechas almacenadas como texto y variables numéricas o categóricas que puedan requerir conversiones.

In [12]:
tipos_datos = pd.DataFrame({
    "Tipo_de_dato": df.dtypes,
    "Valores_no_nulos": df.notnull().sum(),
    "Valores_unicos": df.nunique()
})

tipos_datos

,Tipo_de_dato,Valores_no_nulos,Valores_unicos
Order_ID,str,50000,50000
Order_Date,str,50000,365
Order_Hour,int64,50000,24
Day_of_Week,str,50000,7
Is_Weekend,int64,50000,2
Is_Festival,int64,50000,2
Weather,str,50000,5
Pickup_Zone,str,50000,5
Dropoff_Zone,str,50000,5
Vehicle_Type,str,50000,4


### 4.4 Análisis inicial de variables categóricas

Se inspeccionan los valores únicos de las variables categóricas para identificar su cardinalidad y detectar posibles inconsistencias, diferencias de escritura o categorías anómalas.

In [13]:
columnas_categoricas = df.select_dtypes(
    include=["object", "str", "string", "category"]
).columns

print(f"Cantidad de variables categóricas: {len(columnas_categoricas)}")

for columna in columnas_categoricas:
    print(f"\n--- {columna} ---")
    print(f"Valores únicos: {df[columna].nunique(dropna=False)}")
    print(df[columna].value_counts(dropna=False).head(15))

Cantidad de variables categóricas: 12

--- Order_ID ---
Valores únicos: 50000
Order_ID
ORD000001    1
ORD000002    1
ORD000003    1
ORD000004    1
ORD000005    1
ORD000006    1
ORD000007    1
ORD000008    1
ORD000009    1
ORD000010    1
ORD000011    1
ORD000012    1
ORD000013    1
ORD000014    1
ORD000015    1
Name: count, dtype: int64

--- Order_Date ---
Valores únicos: 365
Order_Date
2025-02-12    170
2025-04-10    169
2025-08-05    168
2025-03-30    167
2025-10-09    166
2025-01-25    165
2025-03-03    165
2025-02-16    164
2025-08-29    164
2025-10-07    163
2025-10-28    162
2025-09-16    162
2025-07-22    162
2025-07-26    161
2025-08-19    160
Name: count, dtype: int64

--- Day_of_Week ---
Valores únicos: 7
Day_of_Week
Tuesday      7264
Sunday       7202
Wednesday    7197
Monday       7160
Thursday     7149
Friday       7050
Saturday     6978
Name: count, dtype: int64

--- Weather ---
Valores únicos: 5
Weather
Clear     23992
Cloudy    11085
Rain       9073
Fog        3413
Storm

### 4.5 Estadística descriptiva de variables numéricas

Se analizan las principales medidas descriptivas de las variables numéricas para obtener una primera aproximación a sus distribuciones, rangos y posibles valores extremos.

En esta etapa los valores atípicos únicamente se identifican como potenciales casos de interés; no se eliminan automáticamente.

In [14]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Order_Hour,50000.0,13.847700,5.525291,0.00,10.00,14.00,19.00,23.00
Is_Weekend,50000.0,0.283600,0.450749,0.00,0.00,0.00,1.00,1.00
Is_Festival,50000.0,0.049100,0.216079,0.00,0.00,0.00,0.00,1.00
Rider_Experience_Years,50000.0,4.439744,2.927752,0.10,2.20,3.80,5.90,15.00
Rider_Rating,50000.0,3.951646,0.294960,3.20,3.70,3.90,4.10,5.00
Restaurant_Rating,50000.0,4.175368,0.356201,3.30,3.90,4.20,4.40,5.00
Order_Items,50000.0,2.652740,1.590359,1.00,1.00,2.00,4.00,7.00
Preparation_Time_Min,50000.0,24.031180,8.348770,5.00,18.00,24.00,30.00,51.00
Road_Distance_km,50000.0,26.201744,12.620079,0.08,16.33,25.62,35.48,70.82
Number_of_Signals,50000.0,9.728920,4.330849,2.00,6.00,9.00,14.00,18.00


## 5. Análisis inicial de la variable objetivo

La variable `Time_taken_min` representa el tiempo de entrega y constituye la variable de interés principal del proyecto.

Antes de analizar sus relaciones con otras características, se estudian su distribución y sus principales estadísticos descriptivos.

In [15]:
TARGET = "Time_taken_min"

df[TARGET].describe()

count    50000.000000
mean        83.868980
std         35.426899
min         10.000000
25%         58.000000
50%         78.000000
75%        102.000000
max        180.000000
Name: Time_taken_min, dtype: float64

In [16]:
print(f"Valores faltantes en {TARGET}: {df[TARGET].isnull().sum():,}")
print(f"Valores únicos: {df[TARGET].nunique():,}")
print(f"Valor mínimo: {df[TARGET].min()}")
print(f"Valor máximo: {df[TARGET].max()}")
print(f"Media: {df[TARGET].mean():.2f}")
print(f"Mediana: {df[TARGET].median():.2f}")

Valores faltantes en Time_taken_min: 0
Valores únicos: 170
Valor mínimo: 10
Valor máximo: 180
Media: 83.87
Mediana: 78.00


## 6. Diagnóstico consolidado de calidad de datos

Se presenta un resumen de los principales indicadores de calidad del dataset original para determinar las transformaciones necesarias y justificar las decisiones de limpieza.

In [17]:
# Resumen general de calidad del dataset original

print("=== DIMENSIONES ===")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

print("\n=== VALORES FALTANTES ===")
print(f"Total de valores faltantes: {df.isna().sum().sum():,}")
print(f"Filas con algún valor faltante: {df.isna().any(axis=1).sum():,}")

print("\n=== DUPLICADOS ===")
print(f"Registros completamente duplicados: {df.duplicated().sum():,}")

print("\n=== VARIABLE OBJETIVO ===")
print(df["Time_taken_min"].describe())

print("\n=== TIPOS DE DATOS ===")
print(df.dtypes.to_string())

print("\n=== COLUMNAS CON VALORES FALTANTES ===")
print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))

=== DIMENSIONES ===
Filas: 50,000
Columnas: 24

=== VALORES FALTANTES ===
Total de valores faltantes: 0
Filas con algún valor faltante: 0

=== DUPLICADOS ===
Registros completamente duplicados: 0

=== VARIABLE OBJETIVO ===
count    50000.000000
mean        83.868980
std         35.426899
min         10.000000
25%         58.000000
50%         78.000000
75%        102.000000
max        180.000000
Name: Time_taken_min, dtype: float64

=== TIPOS DE DATOS ===
Order_ID                          str
Order_Date                        str
Order_Hour                      int64
Day_of_Week                       str
Is_Weekend                      int64
Is_Festival                     int64
Weather                           str
Pickup_Zone                       str
Dropoff_Zone                      str
Vehicle_Type                      str
Rider_Experience_Years        float64
Rider_Rating                  float64
Restaurant_Rating             float64
Cuisine_Type                      str
Order_It

### 6.1. Validación de consistencia de las variables numéricas

Se analizan los valores mínimos, máximos, negativos, iguales a cero y faltantes de las variables numéricas.

El propósito es identificar posibles inconsistencias lógicas que puedan afectar la calidad del análisis exploratorio y del posterior modelado predictivo.

La presencia de valores extremos no implica necesariamente que sean incorrectos, por lo que cualquier decisión de tratamiento deberá justificarse según el significado de cada variable.

In [18]:
# Identificación de columnas numéricas
columnas_numericas = df.select_dtypes(include="number").columns

# Construcción del reporte de consistencia
reporte_numerico = pd.DataFrame({
    "Tipo": df[columnas_numericas].dtypes.astype(str),
    "Mínimo": df[columnas_numericas].min(),
    "Máximo": df[columnas_numericas].max(),
    "Negativos": (df[columnas_numericas] < 0).sum(),
    "Ceros": (df[columnas_numericas] == 0).sum(),
    "Nulos": df[columnas_numericas].isna().sum()
})

reporte_numerico

,Tipo,Mínimo,Máximo,Negativos,Ceros,Nulos
Order_Hour,int64,0.00,23.00,0,463,0
Is_Weekend,int64,0.00,1.00,0,35820,0
Is_Festival,int64,0.00,1.00,0,47545,0
Rider_Experience_Years,float64,0.10,15.00,0,0,0
Rider_Rating,float64,3.20,5.00,0,0,0
Restaurant_Rating,float64,3.30,5.00,0,0,0
Order_Items,int64,1.00,7.00,0,0,0
Preparation_Time_Min,int64,5.00,51.00,0,0,0
Road_Distance_km,float64,0.08,70.82,0,0,0
Number_of_Signals,int64,2.00,18.00,0,0,0


### 6.2. Validación de consistencia de las variables categóricas

Se examinan las categorías registradas para detectar posibles inconsistencias de escritura, espacios innecesarios y valores que representen una misma categoría bajo distintas denominaciones.

Esta evaluación permite determinar si será necesario normalizar algunas variables antes de realizar el análisis exploratorio.

In [19]:
# Inspección de categorías con cardinalidad moderada
for columna in columnas_categoricas:

    cantidad_categorias = df[columna].nunique(dropna=False)

    if cantidad_categorias <= 30:
        print(f"\nVariable: {columna}")
        print(f"Cantidad de categorías: {cantidad_categorias}")

        print(
            df[columna]
            .value_counts(dropna=False)
            .to_string()
        )


Variable: Day_of_Week
Cantidad de categorías: 7
Day_of_Week
Tuesday      7264
Sunday       7202
Wednesday    7197
Monday       7160
Thursday     7149
Friday       7050
Saturday     6978

Variable: Weather
Cantidad de categorías: 5
Weather
Clear     23992
Cloudy    11085
Rain       9073
Fog        3413
Storm      2437

Variable: Pickup_Zone
Cantidad de categorías: 5
Pickup_Zone
Residential    15786
CBD            10997
Commercial      9523
Industrial      7233
Suburban        6461

Variable: Dropoff_Zone
Cantidad de categorías: 5
Dropoff_Zone
Residential    16106
Commercial     10798
CBD             9035
Suburban        7429
Industrial      6632

Variable: Vehicle_Type
Cantidad de categorías: 4
Vehicle_Type
Bike                22943
Scooter             12895
Electric Scooter     8852
Bicycle              5310

Variable: Cuisine_Type
Cantidad de categorías: 9
Cuisine_Type
North Indian    9483
Biryani         7802
Pizza           6661
Chinese         6132
Burger          5198
South India

### 6.3. Resumen consolidado de calidad de datos

Se construye un reporte general que reúne los tipos de datos, valores faltantes, porcentajes de ausencia y cantidad de valores únicos de cada variable.

Este reporte servirá como fundamento para establecer las estrategias de limpieza y transformación que resulten necesarias.

In [20]:
reporte_calidad = pd.DataFrame({
    "Tipo de dato": df.dtypes.astype(str),
    "Valores faltantes": df.isna().sum(),
    "Porcentaje faltante": (df.isna().mean() * 100).round(2),
    "Valores únicos": df.nunique(dropna=False)
})

reporte_calidad = reporte_calidad.sort_values(
    by="Porcentaje faltante",
    ascending=False
)

display(reporte_calidad)

print(f"\nDuplicados completos: {df.duplicated().sum():,}")
print(f"Dimensiones originales: {df.shape}")

,Tipo de dato,Valores faltantes,Porcentaje faltante,Valores únicos
Order_ID,str,0,0.0,50000
Order_Date,str,0,0.0,365
Order_Hour,int64,0,0.0,24
Day_of_Week,str,0,0.0,7
Is_Weekend,int64,0,0.0,2
Is_Festival,int64,0,0.0,2
Weather,str,0,0.0,5
Pickup_Zone,str,0,0.0,5
Dropoff_Zone,str,0,0.0,5
Vehicle_Type,str,0,0.0,4



Duplicados completos: 0
Dimensiones originales: (50000, 24)


## 7. Preparación y limpieza de los datos

A partir de la evaluación de calidad realizada en las secciones anteriores, se establecerán las transformaciones necesarias para preparar el dataset para el análisis exploratorio.

El procedimiento seguirá los siguientes criterios:

- Conservar el dataset original sin modificaciones.
- Identificar y tratar únicamente las inconsistencias verificadas.
- Evitar la eliminación indiscriminada de registros.
- Documentar y justificar las decisiones de limpieza.
- Preservar la información necesaria para el posterior modelado predictivo.

Las transformaciones que requieran estimar parámetros a partir de los datos se realizarán posteriormente dentro del proceso de entrenamiento, utilizando exclusivamente los datos de entrenamiento para evitar fugas de información (*data leakage*).

In [21]:
# Resumen de los resultados de la auditoría

print("=== DIMENSIONES ORIGINALES ===")
print(df.shape)

print("\n=== REGISTROS DUPLICADOS ===")
print(df.duplicated().sum())

print("\n=== VALORES FALTANTES POR VARIABLE ===")
print(df.isna().sum().to_string())

print("\n=== CONSISTENCIA NUMÉRICA ===")
print(reporte_numerico.to_string())

print("\n=== VARIABLE OBJETIVO ===")
print(df["Time_taken_min"].describe().to_string())

=== DIMENSIONES ORIGINALES ===
(50000, 24)

=== REGISTROS DUPLICADOS ===
0

=== VALORES FALTANTES POR VARIABLE ===
Order_ID                      0
Order_Date                    0
Order_Hour                    0
Day_of_Week                   0
Is_Weekend                    0
Is_Festival                   0
Weather                       0
Pickup_Zone                   0
Dropoff_Zone                  0
Vehicle_Type                  0
Rider_Experience_Years        0
Rider_Rating                  0
Restaurant_Rating             0
Cuisine_Type                  0
Order_Items                   0
Restaurant_Load               0
Preparation_Time_Min          0
Road_Distance_km              0
Delivery_Distance_Category    0
Traffic_Level                 0
Number_of_Signals             0
Average_Speed_kmph            0
Delivery_Priority             0
Time_taken_min                0

=== CONSISTENCIA NUMÉRICA ===
                           Tipo  Mínimo  Máximo  Negativos  Ceros  Nulos
Order_Hour   

### 7.2. Creación de una copia de trabajo

Para preservar la integridad del dataset original, se crea una copia independiente sobre la cual se realizarán las transformaciones de limpieza que resulten necesarias.

Las decisiones de tratamiento se fundamentarán en los resultados obtenidos durante la auditoría de calidad, evitando modificaciones arbitrarias o la pérdida innecesaria de información.

In [22]:
# Creación de una copia independiente del dataset original
df_clean = df.copy(deep=True)

print("Copia de trabajo creada correctamente.")
print(f"Dimensiones originales: {df.shape}")
print(f"Dimensiones de la copia: {df_clean.shape}")

# Verificación de que ambos DataFrames son objetos independientes
print(f"¿Son el mismo objeto?: {df is df_clean}")

Copia de trabajo creada correctamente.
Dimensiones originales: (50000, 24)
Dimensiones de la copia: (50000, 24)
¿Son el mismo objeto?: False


### 7.3. Identificación de problemas de calidad

Antes de aplicar transformaciones, se identifican los problemas que podrían requerir intervención, considerando valores faltantes, duplicados y valores numéricos potencialmente inválidos.

La identificación de un problema no implica su eliminación automática. Cada tratamiento deberá evaluarse de acuerdo con el significado de las variables y los objetivos del análisis.

In [23]:
# Resumen de posibles problemas que requieren evaluación

problemas_calidad = pd.DataFrame({
    "Valores faltantes": df_clean.isna().sum(),
    "Porcentaje faltante": (df_clean.isna().mean() * 100).round(2),
    "Valores únicos": df_clean.nunique(dropna=False)
})

display(problemas_calidad)

print("\nRegistros duplicados:", df_clean.duplicated().sum())

print("\nVariables numéricas con valores negativos:")
for columna in df_clean.select_dtypes(include="number").columns:
    cantidad = (df_clean[columna] < 0).sum()
    if cantidad > 0:
        print(f"{columna}: {cantidad}")

,Valores faltantes,Porcentaje faltante,Valores únicos
Order_ID,0,0.0,50000
Order_Date,0,0.0,365
Order_Hour,0,0.0,24
Day_of_Week,0,0.0,7
Is_Weekend,0,0.0,2
Is_Festival,0,0.0,2
Weather,0,0.0,5
Pickup_Zone,0,0.0,5
Dropoff_Zone,0,0.0,5
Vehicle_Type,0,0.0,4



Registros duplicados: 0

Variables numéricas con valores negativos:


### 7.4. Decisiones iniciales de limpieza

La auditoría del dataset original, compuesto por 50.000 registros y 24 variables, permitió identificar las siguientes características:

- No se encontraron valores faltantes.
- No se detectaron registros completamente duplicados.
- No se identificaron valores negativos en las variables numéricas.
- La variable `Order_Hour` presenta valores entre 0 y 23, coherentes con una representación horaria de 24 horas.
- Las variables de distancia, velocidad, preparación y tiempo de entrega presentan valores positivos.

En consecuencia, no se aplicarán eliminaciones de registros ni imputaciones de valores faltantes.

No obstante, será necesario verificar la validez de las fechas, la consistencia de las variables categóricas y la coherencia entre variables relacionadas.

Los valores extremos se conservarán inicialmente hasta determinar si representan observaciones legítimas o inconsistencias.

### 7.5. Validación y conversión de fechas

Se inspecciona el formato original de la variable `Order_Date` y se comprueba si sus valores pueden interpretarse como fechas válidas.

La conversión se realizará únicamente después de verificar el formato y descartar interpretaciones ambiguas.

In [24]:
print("Tipo original:")
print(df_clean["Order_Date"].dtype)

print("\nEjemplos de fechas:")
print(df_clean["Order_Date"].head(10).to_string(index=False))

print("\nCantidad de fechas diferentes:")
print(df_clean["Order_Date"].nunique())

Tipo original:
str

Ejemplos de fechas:
2025-10-28
2025-09-26
2025-01-26
2025-11-19
2025-08-04
2025-01-22
2025-08-19
2025-04-11
2025-01-23
2025-09-19

Cantidad de fechas diferentes:
365


### 7.6. Verificación de coherencia entre variables

Se identifican relaciones entre variables que podrían contener información redundante o inconsistencias.

En particular, se revisará la correspondencia entre fechas y días de la semana, la clasificación de distancias y las características utilizadas para representar las condiciones de entrega.

In [25]:
print("=== DÍA DE LA SEMANA ===")
print(df_clean["Day_of_Week"].value_counts(dropna=False))

print("\n=== FIN DE SEMANA ===")
print(df_clean["Is_Weekend"].value_counts(dropna=False))

print("\n=== CATEGORÍA DE DISTANCIA ===")
print(df_clean["Delivery_Distance_Category"].value_counts(dropna=False))

print("\n=== NIVELES DE TRÁFICO ===")
print(df_clean["Traffic_Level"].value_counts(dropna=False))

=== DÍA DE LA SEMANA ===
Day_of_Week
Tuesday      7264
Sunday       7202
Wednesday    7197
Monday       7160
Thursday     7149
Friday       7050
Saturday     6978
Name: count, dtype: int64

=== FIN DE SEMANA ===
Is_Weekend
0    35820
1    14180
Name: count, dtype: int64

=== CATEGORÍA DE DISTANCIA ===
Delivery_Distance_Category
Long      47238
Medium     2235
Short       527
Name: count, dtype: int64

=== NIVELES DE TRÁFICO ===
Traffic_Level
Low         32567
Moderate    13165
High         3744
Severe        524
Name: count, dtype: int64


### 7.7. Conversión y validación de la variable temporal

La variable `Order_Date` se encuentra originalmente almacenada como texto, con un formato compatible con año-mes-día.

Se procede a convertirla al tipo `datetime`, verificando la existencia de fechas inválidas y el período temporal cubierto por los registros.

Posteriormente, se evaluará su coherencia con las variables `Day_of_Week` e `Is_Weekend`.

In [26]:
# Conversión explícita de fechas
df_clean["Order_Date"] = pd.to_datetime(
    df_clean["Order_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Tipo de dato:")
print(df_clean["Order_Date"].dtype)

print("\nFechas inválidas:")
print(df_clean["Order_Date"].isna().sum())

print("\nPrimera fecha:")
print(df_clean["Order_Date"].min())

print("\nÚltima fecha:")
print(df_clean["Order_Date"].max())

Tipo de dato:
datetime64[us]

Fechas inválidas:
0

Primera fecha:
2025-01-01 00:00:00

Última fecha:
2025-12-31 00:00:00


### 7.8. Validación de coherencia temporal

Se compara el día de la semana declarado en `Day_of_Week` con el día calculado a partir de `Order_Date`.

Asimismo, se verifica la coherencia entre la fecha y el indicador binario `Is_Weekend`.

Estas comprobaciones permiten identificar posibles contradicciones entre variables relacionadas.

In [27]:
# Día de la semana calculado a partir de la fecha
dia_calculado = df_clean["Order_Date"].dt.day_name()

# Comparación con la variable original
inconsistencias_dia = (
    df_clean["Order_Date"].notna()
    & (dia_calculado != df_clean["Day_of_Week"])
)

print("Inconsistencias en Day_of_Week:")
print(inconsistencias_dia.sum())

# Cálculo del indicador de fin de semana
fin_semana_calculado = (
    df_clean["Order_Date"].dt.dayofweek >= 5
).astype(int)

inconsistencias_fin_semana = (
    df_clean["Order_Date"].notna()
    & (fin_semana_calculado != df_clean["Is_Weekend"])
)

print("\nInconsistencias en Is_Weekend:")
print(inconsistencias_fin_semana.sum())

Inconsistencias en Day_of_Week:
0

Inconsistencias en Is_Weekend:
0


### 7.9. Evaluación de las categorías de distancia

Se analiza la relación entre `Road_Distance_km` y `Delivery_Distance_Category` para identificar los rangos de distancia asociados con cada categoría.

El objetivo es comprender la clasificación existente y determinar si presenta solapamientos o inconsistencias.

In [28]:
resumen_distancias = (
    df_clean
    .groupby("Delivery_Distance_Category")["Road_Distance_km"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(2)
)

display(resumen_distancias)

,count,min,median,mean,max
Delivery_Distance_Category,,,,,
Long,47238,7.01,26.63,27.46,70.82
Medium,2235,3.01,5.39,5.27,7.00
Short,527,0.08,2.05,1.97,3.00


### 7.10. Conclusiones de la evaluación y preparación de datos

La auditoría realizada sobre el dataset de 50.000 registros y 24 variables permitió verificar una calidad inicial satisfactoria.

No se identificaron valores faltantes, registros completamente duplicados ni valores negativos en las variables numéricas. Asimismo, la conversión de `Order_Date` al tipo `datetime` se realizó sin producir fechas inválidas.

El conjunto de datos comprende el período del 1 de enero al 31 de diciembre de 2025. Se comprobó que los días de la semana y los indicadores de fin de semana coinciden con las fechas registradas.

Por otra parte, las categorías de distancia presentan intervalos observados no superpuestos: Short (hasta 3 km), Medium (entre 3,01 y 7 km) y Long (desde 7,01 km).

En consecuencia, no fue necesario eliminar registros ni imputar valores faltantes. Se conservan las observaciones originales y se realiza únicamente la conversión de la variable temporal.

Se mantiene como aspecto relevante para el análisis exploratorio la elevada concentración de pedidos en la categoría Long.

Los posibles valores atípicos serán evaluados durante el EDA, sin eliminarlos automáticamente.

### 7.11. Exportación del dataset preparado

Se exporta una versión del dataset luego de completar las validaciones iniciales y la conversión de la variable temporal.

Este archivo se utilizará como entrada para el análisis exploratorio, conservando intacto el CSV original.

Las transformaciones que requieran aprender parámetros estadísticos se reservarán para el posterior pipeline de entrenamiento, evitando fugas de información entre los conjuntos de entrenamiento, validación y prueba.

In [29]:
# Ruta de salida del dataset preparado
OUTPUT_PATH = Path("../data/processed/food_delivery_prepared.csv")

# Crear directorio si no existe
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Exportar sin agregar el índice de pandas
df_clean.to_csv(
    OUTPUT_PATH,
    index=False,
    date_format="%Y-%m-%d"
)

print("Dataset preparado guardado correctamente.")
print(f"Ruta: {OUTPUT_PATH}")
print(f"Dimensiones: {df_clean.shape}")
print(f"Archivo existente: {OUTPUT_PATH.exists()}")

Dataset preparado guardado correctamente.
Ruta: ..\data\processed\food_delivery_prepared.csv
Dimensiones: (50000, 24)
Archivo existente: True
